This is a little memory and runtime profiler for checkpoints. First, let's define some constants.

In [ ]:
from time import time

CHECKPOINT_PATH = "./checkpoints/checkpoint.pt"
DEVICE = "cpu"  # Can be "cpu" or "cuda."
BATCH_SIZE = 1
TRACE_PATH = f"./exports/traces/trace-{time()}.json"

Then we'll load the checkpoint and instantiate the model.

In [ ]:
import torch

from src.esmc_protein_function.model import ESMCProteinFunction

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=True,
)

model = ESMCProteinFunction(**checkpoint["model_args"])

model.load_state_dict(checkpoint["model"])

model.remove_fake_quantized_tensors()

model = model.to(DEVICE)

print("Model loaded successfully")

Now let's make some fake data.

In [ ]:
# Make random integer vectors to simulate input sequences. [0, 20] is the range of token IDs in the ESM tokenizer.

This next block we'll run a forward pass on the fake data within the context of the profiler.

In [ ]:

from torch.profiler import profile, record_function, ProfilerActivity

match DEVICE:
    case "cpu":
        activities = [ProfilerActivity.CPU]
    case "cuda":
        activities = [ProfilerActivity.CUDA]
    case _:
        raise ValueError(f"Unsupported device: {DEVICE}")

with profile(activities=activities, profile_memory=True, record_shapes=True) as profiler:
    with record_function("model_inference"):
        y_pred = model.predict_all(x)

Now let's print out the data that the profiler collected for us.

In [ ]:
print(profiler.key_averages().table())

Finally, we'll export a Chrome trace so we can view it in a Chromium-compatible web browser.

In [ ]:
profiler.export_chrome_trace(TRACE_PATH)